In [ ]:
import pandas as pd
import numpy as np
import glob
import os

path = './data' 
all_files = glob.glob(os.path.join(path, "*.csv"))


# Read each CSV and append it to a list
df_list = []
for file in all_files:
    # low_memory=False prevents mixed-datatype warnings during loading
    df = pd.read_csv(file, low_memory=False)
    df_list.append(df)

# Combine everything
master_df = pd.concat(df_list, ignore_index=True)
print(f"Data loaded successfully! Total rows: {len(master_df)}")

Found 8 files. Loading...
Data loaded successfully! Total rows: 2830743


In [ ]:
# This strips the spaces so code doesn't crash later
master_df.columns = master_df.columns.str.strip()

print("Columns cleaned. Here are the first 10 columns:")
print(master_df.columns[:10].tolist())

Columns cleaned. Here are the first 10 columns:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']


In [ ]:
print(f"Rows before cleaning: {len(master_df)}")

# 1. Convert Infinity to NaN so Pandas can identify them as missing
master_df.replace([np.inf, -np.inf], np.nan, inplace=True)

missing_counts = master_df.isna().sum()
print("\n--- The Culprits (Columns with missing data) ---")
print(missing_counts[missing_counts > 0])
print("----------------------------------------------\n")

# If a column is more than 50% broken/missing, delete the whole column.
threshold = len(master_df) * 0.5
master_df.dropna(axis=1, thresh=threshold, inplace=True)

# drop the individual rows that have a stray NaN (usually caused by Infinity).
master_df.dropna(axis=0, inplace=True)

print(f"Rows after cleaning: {len(master_df)}")

Rows before cleaning: 2830743

--- The Culprits (Columns with missing data) ---
Flow Bytes/s         2867
Flow Packets/s       2867
Active Max         445909
Idle Max           286467
Idle Max          2544276
Active Max        2384834
dtype: int64
----------------------------------------------

Rows after cleaning: 2096135


In [4]:
# Convert 'BENIGN' to 0 (Normal) and everything else to 1 (Attack)
master_df['Attack_Flag'] = master_df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

# Drop the old text-based Label column
master_df.drop(columns=['Label'], inplace=True)

print("New label distribution:")
print(master_df['Attack_Flag'].value_counts())

New label distribution:
Attack_Flag
0    1712215
1     383920
Name: count, dtype: int64


In [5]:
# Save the finalized, clean dataset to a new CSV file
output_filename = 'CIC_IDS2017_Clean.csv'
master_df.to_csv(output_filename, index=False)